In [31]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
import numpy as np
import matplotlib.pyplot as plt
import math
import time
from src.model import FiLMResNet2In, flatten_last
from src.normalizer import RunningMeanStd
from src.envpacker import packenv, packenv_batch
from src.utils import transition_ability_batched, update_ability_history, _summarize_tensor
import torch
from torch import nn
import torch.nn.functional as F
from tqdm import tqdm
import wandb
import os
import pickle

## 1. Configs

In [33]:
# Training configs
AGENTS = 50     # number of agents
LEARNING_RATE = 1e-3
TRAINING_STEPS = 30000
BATCH_SIZE = 10
DISPLAY_STEP = 100 # For visualization
TRAIN_STEP_INTERVAL = 2 # Interval of steps between training episodes
LAMBDA_AUX = 1.0 # Loss weighting parameters
LAMBDA_FOC = 1.0
MAX_HISTORY_LEN = 50


# Bewley model parameters
theta = 1 # CRRA
beta = 0.975 # Discount factor
A = 1 # Technology parameter
alpha = 0.33 # Capital share of income
gamma = 2 # Inverse Frisch elasticity
########################################### (MiLF inputs)
r = 0.06 # Interest rate (Return to savings)
w = 1 # Wage rate (Return to labor)
delta = 0.04 # Depreciation rate of capital
TAX_PARAMS = {
    "tax_consumption": 0.065,          # Consumption tax (fixed)
    "tax_income": 0.5,                # Tax on labor income
    "income_tax_elasticity": 0.5,     # Elasticity of labor supply w.r.t. after-tax income
    "saving_tax_elasticity": 0.5,     # Elasticity of savings w.r.t. after-tax income
    "tax_saving": 0.5                 # Tax on interest income
}
###########################################
p = 2.2e-6
q = 0.99

# shock parameters
# log e' = rho_v * log e + sigma_v * epsilon, epsilon ~ N(0,1)
rho_v = 0.95 # persistence of ability shock
sigma_v = 0.2 # std of ability shock
v_bar = 1.5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Experiment setup ---
EXP_NAME = "exp_transition"
RUN_NAME = "run"
# --- Directory structure ---
BASE_DIR = os.path.join("./checkpoints", EXP_NAME, RUN_NAME)
WEIGHT_DIR = os.path.join(BASE_DIR, "weights")
STATE_DIR  = os.path.join(BASE_DIR, "states")

os.makedirs(WEIGHT_DIR, exist_ok=True)
os.makedirs(STATE_DIR, exist_ok=True)
final_weight_path = os.path.join(WEIGHT_DIR, "model_final.pt")
final_state_path = os.path.join(STATE_DIR, "final_state.pkl")


In [34]:
# Bounds of shock 
v_min = math.exp(-2 * sigma_v  / math.sqrt(1-rho_v**2))
v_max = math.exp( 2 * sigma_v  / math.sqrt(1-rho_v**2))

## 2. Helper functions and classes

In [35]:
def mean_across_agents(x): # Since agent number is fix, thus we can use mean instead of sum
    return torch.mean(x, dim=1, keepdim=True)


def calculate_price(savings, ability, labor, A=1.0, alpha=0.36):
    # mean over agent dimension (dim=1)
    savings_agg = savings.mean(dim=1, keepdim=True)
    labor_eff_agg = (labor * ability).mean(dim=1, keepdim=True)
    
    # avoid zero or negative denominators
    labor_eff_agg = torch.clamp(labor_eff_agg, min=1e-8)
    
    ratio = savings_agg / labor_eff_agg
    ratio = torch.clamp(ratio, min=1e-8)
    
    wage = A * (1 - alpha) * (ratio ** alpha)
    ret = A * alpha * (ratio ** (alpha - 1))  # corrected exponent
    return wage, ret


def taxfunc(ibt, abt, taxparams=TAX_PARAMS):
    it = ibt - (1 - taxparams["tax_income"]) * (ibt**(1-taxparams["income_tax_elasticity"])/(1-taxparams["income_tax_elasticity"])) # individual after tax income
    at = abt - ((1-taxparams["tax_saving"])/(1-taxparams["saving_tax_elasticity"])) * (abt**(1-taxparams["saving_tax_elasticity"])) # individual after tax saving
    return it, at


def calculate_moneydisposable(
    wage, ret, ability, labor, savings, delta, training_step, verbose=True
):
    # --- coerce to tensors ---
    if not torch.is_tensor(savings): savings = torch.tensor(savings, dtype=torch.float32)
    if not torch.is_tensor(wage):    wage    = torch.tensor(wage,    dtype=torch.float32)
    if not torch.is_tensor(ret):     ret     = torch.tensor(ret,     dtype=torch.float32)
    if not torch.is_tensor(ability): ability = torch.tensor(ability, dtype=torch.float32)
    if not torch.is_tensor(labor):   labor   = torch.tensor(labor,   dtype=torch.float32)
    if not torch.is_tensor(delta):   delta   = torch.tensor(delta,   dtype=torch.float32)

    # keep ability nonnegative-ish
    ability = torch.clamp(ability, min=0.1)

    # --- compute before-tax income ---
    ibt = wage * labor * ability + (1 - delta + ret) * savings  # before-tax income

    # --- helper: stats printer ---
    def _summarize(x, name):
        x = x if torch.is_tensor(x) else torch.tensor(x, dtype=torch.float32)
        with torch.no_grad():
            # flatten for robust counting
            xf = x.reshape(-1).to(torch.float32)
            isnan = torch.isnan(xf)
            isinf = torch.isinf(xf)
            valid = ~(isnan | isinf)
            xv = xf[valid] if valid.any() else torch.tensor([], dtype=xf.dtype)

            def _safe_stat(t, fn, default=float('nan')):
                return fn(t).item() if t.numel() > 0 else default

            pos = (xv > 0).sum().item()
            neg = (xv < 0).sum().item()
            zer = (xv == 0).sum().item()
            tot = xv.numel()

            minv = _safe_stat(xv, torch.min)
            maxv = _safe_stat(xv, torch.max)
            mean = _safe_stat(xv, torch.mean)

            # fractions (avoid 0 division)
            def frac(n):
                return (n / tot) if tot > 0 else float('nan')

            print(f"{name:>12s}: shape={tuple(x.shape)}, dtype={x.dtype}")
            print(f"{'':12s}  min={minv:.4e}  max={maxv:.4e}  mean={mean:.4e}")
            print(f"{'':12s}  pos={pos} ({frac(pos):.2%})  neg={neg} ({frac(neg):.2%})  zero={zer} ({frac(zer):.2%})")
            print(f"{'':12s}  NaN={isnan.sum().item()}  Inf={isinf.sum().item()}  valid={tot}")

    # --- boxed logging block (always includes positives) ---
    if verbose:
        bar = "+" * 66
        print("\n" + bar)
        print(f"+  calculate_moneydisposable b4 tax func| step={training_step:<8} ".ljust(65) + "+")
        print(bar)
        _summarize(wage,    "wage")
        _summarize(ret,     "ret")
        _summarize(ability, "ability")
        _summarize(labor,   "labor")
        _summarize(delta,   "delta")
        _summarize(savings, "savings")
        _summarize(ibt,     "ibt")
        print(bar + "\n")

    # --- compute after-tax components ---
    it, at = taxfunc(ibt=ibt, abt=savings)
    money_disposable = (ibt-it) + (savings-at) 
    return money_disposable, ibt



def output_transform(savings, money_disposable):

    # The a here is saving rate coming from the NN output
    consumption = money_disposable * (1 - savings)
    savings = money_disposable * savings    
    return consumption, savings


def fbloss(savings_ratio, mutilpier):
    R1 = savings_ratio 
    R2 = (1-mutilpier) 
    return torch.mean((R1+R2-torch.sqrt(R1**2+R2**2))**2)

def auxiloss(c0, c1_A, c1_B, savings2_A, savings2_B, mu_t0,
             ibt_A, ibt_B, ret0, theta, beta=1.0, taxparams=None, eps=1e-8):
    """
    Numerically robust auxiliary loss function.
    """

    # --- default params ---
    if taxparams is None:
        taxparams = TAX_PARAMS

    # --- unpack ---
    e_inc = taxparams["income_tax_elasticity"]
    e_sav = taxparams["saving_tax_elasticity"]
    t_inc = taxparams["tax_income"]
    t_sav = taxparams["tax_saving"]

    # --- ensure all are tensors on same device ---
    tensors = [c0, c1_A, c1_B, savings2_A, savings2_B, mu_t0, ibt_A, ibt_B, ret0]
    device = next(t.device for t in tensors if torch.is_tensor(t))
    dtype = next(t.dtype for t in tensors if torch.is_tensor(t))

    def to_tensor(x):
        return x if torch.is_tensor(x) else torch.tensor(x, device=device, dtype=dtype)

    c0, c1_A, c1_B, savings2_A, savings2_B, mu_t0, ibt_A, ibt_B, ret0 = map(to_tensor, tensors)

    # --- stabilize division and exponentiation ---
    def safe_pow(base, exp):
        base_safe = torch.clamp(base, min=eps)
        return torch.exp(exp * torch.log(base_safe))

    def safe_ratio(num, denom):
        denom_safe = torch.clamp(denom, min=eps)
        return num / denom_safe

    # --- main terms (A and B branches) ---
    cons_ratio_A = safe_ratio(c1_A, c0)
    cons_ratio_B = safe_ratio(c1_B, c0)

    inc_term_A = safe_pow(ibt_A, e_inc)
    inc_term_B = safe_pow(ibt_B, e_inc)

    sav_term_A = safe_pow(savings2_A, e_sav)
    sav_term_B = safe_pow(savings2_B, e_sav)

    aux_1 = beta * safe_pow(cons_ratio_A, -theta) * (
        ret0 * (1 - t_inc) * inc_term_A
    ) + (1 - t_sav) * sav_term_A - mu_t0

    aux_2 = beta * safe_pow(cons_ratio_B, -theta) * (
        ret0 * (1 - t_inc) * inc_term_B
    ) + (1 - t_sav) * sav_term_B - mu_t0

    # --- optional NaN debug (silent) ---
    if torch.isnan(aux_1).any() or torch.isnan(aux_2).any():
        print("⚠️ NaN detected in aux_1 or aux_2")
        for name, t in [("c0", c0), ("c1_A", c1_A), ("ibt_A", ibt_A), ("savings2_A", savings2_A)]:
            if torch.isnan(t).any():
                print(f"  -> {name} contains NaN")

    # --- stabilize before mean ---
    aux_1 = torch.nan_to_num(aux_1, nan=0.0, posinf=1e6, neginf=-1e6)
    aux_2 = torch.nan_to_num(aux_2, nan=0.0, posinf=1e6, neginf=-1e6)

    loss = torch.mean(aux_1 * aux_2)

    # clamp again to avoid inf loss
    if not torch.isfinite(loss):
        loss = torch.tensor(1e6, device=device, dtype=dtype)
        print("⚠️ Loss overflow → replaced with 1e6")

    # for name, t in {
    # "c0": c0, "c1_A": c1_A, "ibt_A": ibt_A,
    # "savings2_A": savings2_A, "ret0": ret0
    # }.items():
    #     print(f"{name}: mean={t.mean():.3e}, std={t.std():.3e}, "
    #         f"min={t.min():.3e}, max={t.max():.3e}")
    return loss

# 仔細檢查這個loss
# def auxiloss(c0, c1_A, c1_B, savings2_A, savings2_B, mu_t0,
#              ibt_A, ibt_B, ret0, theta, taxparmas = TAX_PARAMS):

#     aux_1 = beta * (c1_A / c0) ** (-theta) * \
#         (ret0 * (1 - taxparmas["tax_income"]) * (ibt_A) ** taxparmas["income_tax_elasticity"]) + \
#         (1 - taxparmas["tax_saving"]) * savings2_A ** taxparmas["saving_tax_elasticity"] - \
#         mu_t0

#     aux_2 = beta * (c1_B / c0) ** (-theta) * \
#         (ret0 * (1 - taxparmas["tax_income"]) * (ibt_B) ** taxparmas["income_tax_elasticity"]) + \
#         (1 - taxparmas["tax_saving"]) * savings2_B ** taxparmas["saving_tax_elasticity"] - \
#         mu_t0

#     print(f"aux_1 : {aux_1} / aux_2 : {aux_2}")

#     return torch.mean(aux_1*aux_2)

def laborfocloss(labor, consumption, ibt, wage, ability, taxparams=TAX_PARAMS):
    loss_foc = -labor**gamma + \
        ((consumption**-(theta)) / (1+ taxparams["tax_saving"])) * \
            (wage*ability*(1-(1-taxparams["tax_income"]*ibt**(-taxparams["income_tax_elasticity"]))))
    return torch.mean(torch.abs(loss_foc))



In [36]:
def auxiloss_stable_v2(
    c0, c1_A, c1_B, savings2_A, savings2_B, mu_t0,
    ibt_A, ibt_B, ret0, theta, beta=1.0,
    taxparams=None, eps=1e-8, scale=1e4, mode="log"
):
    """
    Numerically stable auxiliary loss for macro-RL equilibrium consistency.

    Parameters
    ----------
    c0, c1_A, c1_B, savings2_A, savings2_B, mu_t0, ibt_A, ibt_B, ret0 : Tensor
        Input tensors (shape: (B, A, 1) or broadcastable).
    theta : float
        Risk aversion / curvature parameter.
    beta : float, default=1.0
        Discount factor.
    taxparams : dict
        Contains "income_tax_elasticity", "saving_tax_elasticity", "tax_income", "tax_saving".
    eps : float
        Numerical floor to avoid log(0) or division by zero.
    scale : float
        Normalization factor for all flow variables.
    mode : {"linear", "log"}
        - "linear":  computes mean(aux_1 * aux_2)
        - "log":     computes mean(log1p(abs(aux_1 * aux_2)))

    Returns
    -------
    loss : torch.Tensor (scalar)
    """

    if taxparams is None:
        taxparams = TAX_PARAMS

    # --- unpack tax params ---
    e_inc = taxparams["income_tax_elasticity"]
    e_sav = taxparams["saving_tax_elasticity"]
    t_inc = taxparams["tax_income"]
    t_sav = taxparams["tax_saving"]

    # --- ensure tensor dtype/device consistency ---
    tensors = [c0, c1_A, c1_B, savings2_A, savings2_B,
               mu_t0, ibt_A, ibt_B, ret0]
    device = next(t.device for t in tensors if torch.is_tensor(t))
    dtype = next(t.dtype for t in tensors if torch.is_tensor(t))

    def to_tensor(x):
        return x if torch.is_tensor(x) else torch.tensor(x, device=device, dtype=dtype)

    c0, c1_A, c1_B, savings2_A, savings2_B, mu_t0, ibt_A, ibt_B, ret0 = map(to_tensor, tensors)

    # --- normalize & stabilize ---
    def normalize(x):
        x = torch.abs(x / scale)
        return torch.clamp(x, min=eps)

    c0 = normalize(c0)
    c1_A = normalize(c1_A)
    c1_B = normalize(c1_B)
    ibt_A = normalize(ibt_A)
    ibt_B = normalize(ibt_B)
    savings2_A = normalize(savings2_A)
    savings2_B = normalize(savings2_B)

    # --- helper safe power ---
    def safe_pow(base, exp):
        return torch.exp(exp * torch.log(base + eps))

    # --- compute terms (branch A / B) ---
    cons_ratio_A = torch.clamp(c1_A / c0, min=eps, max=1e3)
    cons_ratio_B = torch.clamp(c1_B / c0, min=eps, max=1e3)

    inc_term_A = safe_pow(ibt_A, e_inc)
    inc_term_B = safe_pow(ibt_B, e_inc)
    sav_term_A = safe_pow(savings2_A, e_sav)
    sav_term_B = safe_pow(savings2_B, e_sav)

    aux_1 = beta * safe_pow(cons_ratio_A, -theta) * (
        ret0 * (1 - t_inc) * inc_term_A
    ) + (1 - t_sav) * sav_term_A - mu_t0

    aux_2 = beta * safe_pow(cons_ratio_B, -theta) * (
        ret0 * (1 - t_inc) * inc_term_B
    ) + (1 - t_sav) * sav_term_B - mu_t0

    # --- nan/inf cleanup ---
    aux_1 = torch.nan_to_num(aux_1, nan=0.0, posinf=1e6, neginf=-1e6)
    aux_2 = torch.nan_to_num(aux_2, nan=0.0, posinf=1e6, neginf=-1e6)

    # --- compute loss ---
    if mode == "log":
        loss = torch.mean(torch.log1p(torch.abs(aux_1 * aux_2)))
    elif mode == "linear":
        loss = torch.mean(aux_1 * aux_2)
    else:
        raise ValueError("mode must be 'log' or 'linear'")

    # --- stabilize loss output ---
    if not torch.isfinite(loss):
        print("⚠️ Loss overflow → replacing with finite fallback (1e6)")
        loss = torch.tensor(1e6, device=device, dtype=dtype)

    return loss

def detach_state_dict(state_dict):
    detached = {}
    for key, value in state_dict.items():
        if torch.is_tensor(value):
            # Create a completely new tensor, breaking all references
            detached[key] = value.detach().clone()
        elif isinstance(value, dict):
            detached[key] = detach_state_dict(value)
        else:
            detached[key] = value
    return detached

In [37]:
def laborfocloss_stable(
    labor, consumption, ibt, wage, ability,
    taxparams=TAX_PARAMS,
    gamma=1.0,
    theta=1.0,
    eps=1e-8,
    clip_val=1e3,
    scale=1.0,
    mode="log"
):
    """
    Numerically stable labor first-order-condition loss.

    F.O.C. form (target = 0):
        -labor**γ + (consumption**(-θ)/(1+τ_s)) * [wage*ability*(1-(1-τ_i) * ibt**(-ε_i))]

    Robust version keeps equilibrium point identical but clamps extreme values.

    Parameters
    ----------
    eps : float
        Floor for divisions / logs to avoid NaNs.
    clip_val : float
        Clamp magnitude of intermediate terms.
    scale : float
        Optional normalization factor for economic magnitudes.
    mode : {"linear","log"}
        "linear" → mean(|loss_foc|)
        "log"    → mean(log1p(|loss_foc|)) for robust smoothing.
    """

    # --- unpack ---
    t_inc = taxparams["tax_income"]
    t_sav = taxparams["tax_saving"]
    e_inc = taxparams["income_tax_elasticity"]

    # --- helper: safe power ---
    def safe_pow(base, exp):
        base = torch.clamp(torch.abs(base / scale), min=eps, max=clip_val)
        return torch.exp(exp * torch.log(base + eps))

    # --- compute terms safely ---
    labor_term = -safe_pow(labor, gamma)
    cons_term  = safe_pow(consumption, -theta) / (1.0 + t_sav)

    ibt_term = safe_pow(ibt, -e_inc)
    # tax_factor = (1-(1.0 - t_inc) * ibt_term)
    tax_factor = (1.0 - t_inc) * ibt_term

    # multiply with clamping to avoid overflow
    prod_term = torch.clamp(wage * ability * tax_factor, min=-clip_val, max=clip_val)

    # --- main FOC residual ---
    loss_foc = labor_term + cons_term * prod_term

    # --- clean up NaN / Inf ---
    loss_foc = torch.nan_to_num(loss_foc, nan=0.0, posinf=clip_val, neginf=-clip_val)

    # --- aggregate robustly ---
    if mode == "log":
        loss = torch.mean(torch.log1p(torch.abs(loss_foc)))
    elif mode == "linear":
        loss = torch.mean(torch.abs(loss_foc))
    else:
        raise ValueError("mode must be 'linear' or 'log'")

    if not torch.isfinite(loss):
        print("⚠️ laborfocloss overflow → setting fallback 1e6")
        loss = torch.tensor(1e6, device=loss_foc.device, dtype=loss_foc.dtype)

    return loss

In [38]:
state_dim = 2*AGENTS + 2 # two state variables for each agent + 2 individual variables
cond_dim = 5 # exogenous variables for all agents in all worlds(Batch)
model = FiLMResNet2In(state_dim=state_dim, cond_dim=cond_dim,
                        hidden_dim=128, num_res_blocks=3, output_dim=3, dropout=0.1).to(DEVICE)

In [39]:
def initial_state(required_batch_size, tax_params_dict=TAX_PARAMS, device = DEVICE):
    # 隨機產生初始資產與儲蓄
    moneydisposable = np.random.lognormal(0.1, 2.0, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)
    savings = np.random.lognormal(0.1, 2.0, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)

    # Initial productivity
    ability = np.random.lognormal(0.5, 1.5, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)
    # ability = ability / np.mean(ability, axis=1, keepdims=True)


    # superstar 標誌 (v1 對應一組, v2 對應一組)
    is_superstar_v1 = np.zeros((required_batch_size, AGENTS), dtype=bool)
    is_superstar_v2 = np.zeros((required_batch_size, AGENTS), dtype=bool)

    # 稅制參數轉為 tensor
    tax_params = torch.tensor(list(tax_params_dict.values()), dtype=torch.float32)
    tax_params = tax_params.repeat(required_batch_size, 1)

    # 轉為 tensor
    moneydisposable_t = torch.tensor(moneydisposable, dtype=torch.float32)
    savings_t = torch.tensor(savings, dtype=torch.float32)
    ability_t = torch.tensor(ability, dtype=torch.float32)
    is_superstar_vA = torch.tensor(is_superstar_v1, dtype=torch.bool)
    is_superstar_vB = torch.tensor(is_superstar_v2, dtype=torch.bool)
    tax_params_t = tax_params

    # 回傳字典
    return {
        "moneydisposable": moneydisposable_t,
        "savings":savings_t,
        "ability":ability_t,
        "is_superstar_vA": is_superstar_vA,
        "is_superstar_vB": is_superstar_vB,
        "tax_params": tax_params_t,
        "ret": r
    }


In [78]:
# def build_inputs(moneydisposable, v, tax_params, device=DEVICE):
#     """
#     回傳:
#       features : (B, A, 2A + 2)          # 給模型輸入
#       condi    : (B, A, Z)              # 稅制條件
#       env_info : dict                   # 僅供環境轉移使用，不進模型
#     """
#     B, A = moneydisposable.shape

#     # (B, Z) -> (B, A, Z)
#     condi = tax_params.unsqueeze(1).expand(-1, A, -1)

#     # (B, 2A) -> (B, A, 2A)
#     sum_info = torch.cat([moneydisposable, v], dim=1)         # (B, 2A)
#     sum_info_rep = sum_info.unsqueeze(1).expand(-1, A, -1)    # (B, A, 2A)

#     # (B, A, 1) × 2
#     money_self = moneydisposable.unsqueeze(-1)  # (B, A, 1)
#     v_self     = v.unsqueeze(-1)                # (B, A, 1)

#     # 給模型的 features
#     features = torch.cat([sum_info_rep, money_self, v_self], dim=2)  # (B, A, 2A+2)

#     return features.to(DEVICE), condi.to(DEVICE)

def build_inputs(moneydisposable, ability, tax_params, device=DEVICE, eps=1e-8):
    """
    回傳:
      features : (B, A, 2A + 2)   # 給模型輸入
      condi    : (B, A, Z)        # 稅制條件
    """
    # ---- 1. 保證都是 tensor ----
    moneydisposable = torch.as_tensor(moneydisposable, dtype=torch.float32, device=device)
    ability         = torch.as_tensor(ability,         dtype=torch.float32, device=device)
    tax_params      = torch.as_tensor(tax_params,      dtype=torch.float32, device=device)

    B, A = moneydisposable.shape

    # ---- 2. Min–Max 標準化 (逐 batch) ----
    m_min = moneydisposable.min(dim=1, keepdim=True)[0]
    m_max = moneydisposable.max(dim=1, keepdim=True)[0]
    moneydisposable = (moneydisposable - m_min) / (m_max - m_min + eps)

    # ---- 3. 原本流程 ----
    # (B, Z) -> (B, A, Z)
    condi = tax_params.unsqueeze(1).expand(-1, A, -1)   # tax_params: (B, Z)

    # (B, 2A) -> (B, A, 2A)
    sum_info = torch.cat([moneydisposable, ability], dim=1)     # (B, 2A)
    sum_info_rep = sum_info.unsqueeze(1).expand(-1, A, -1)      # (B, A, 2A)

    # (B, A, 1) × 2
    money_self   = moneydisposable.unsqueeze(-1)  # (B, A, 1)
    ability_self = ability.unsqueeze(-1)          # (B, A, 1)

    features = torch.cat([sum_info_rep, money_self, ability_self], dim=2)  # (B, A, 2A+2)

    return features, condi


In [41]:
def run_network_given_state(state, model, brach=None):
    
    # 把整個state agent 需要作動的部分取出來

    if not brach:
        agent_state = build_inputs(
            moneydisposable=state["moneydisposable"],
            ability=state["ability"],
            tax_params=state["tax_params"]
        )

    else:
        # print(state["moneydisposable"])
        agent_state = build_inputs(
            moneydisposable=state["moneydisposable"],
            ability=state[f"ability_{brach}"],
            tax_params=state["tax_params"]
        )

    acts = [torch.sigmoid, lambda x: F.softplus(x) + 1e-6, torch.sigmoid]
    out = model(agent_state[0], agent_state[1])
    savings_t1, mu_t0, labor_t0 = [acts[i](out[..., i]) for i in range(out.shape[-1])]
    savings_t1, mu_t0, labor_t0 = savings_t1.squeeze(-1), mu_t0.squeeze(-1), labor_t0.squeeze(-1)

    return {
        "savings_t1": savings_t1,
        "mu_t0": mu_t0,
        "labor_t0": labor_t0,
    }



In [42]:
def part_transition_transform(state, model_out, training_step, branch=None, is_init=False, device=DEVICE):
    """
    Compute the partial transition transform for the current state and model output.
    Now supports device placement (CPU/GPU) for all intermediate tensors.
    """

    # --- Ensure everything is on the correct device ---
    # Move all tensor-like inputs to the same device
    if device is not None:
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)
        for key, value in model_out.items():
            if torch.is_tensor(value):
                model_out[key] = value.to(device)

    # --- Current wage and return ---
    ability_key = "ability" if not branch else f"ability_{branch}"
    ability = state[ability_key]

    wage, ret = calculate_price(
        savings=state["savings"],
        ability=ability,
        labor=model_out["labor_t0"],
    )

    # --- Disposable money and interest before tax ---
    # if is_init, interest rate comes from previous period
    # _summarize_tensor(state["savings"], "savings(raw)")
    money_disposable, ibt = calculate_moneydisposable(
        wage=wage, # 當期的wage 
        ret=state["ret"], # 上期的return
        ability=ability,
        labor=model_out["labor_t0"],
        savings=state["savings"],
        delta=delta,
        training_step = training_step,
        verbose=False
    )
    # 這裡money 是負的
    # print(f'Money disposable f{money_disposable[0]}')
    # --- Consumption and next-period savings ---
    # _summarize_tensor(money_disposable, "money_disposable(before transform)")
    consumption, savings = output_transform(
        savings=model_out["savings_t1"],
        money_disposable=money_disposable,
    )
    # _summarize_tensor(savings, "savings(after transform)")

    # --- Move outputs to the correct device ---
    result = {
        "moneydisposable": money_disposable,
        "savings": savings,               # savings for next period
        "consumption": consumption,       # consumption for current period
        "wage": wage,
        "ret": ret,
        "ibt": ibt,
        "tax_params": state["tax_params"],
    }

    if device is not None:
        for key, value in result.items():
            if torch.is_tensor(value):
                result[key] = value.to(device)

    return result


In [ ]:
def run_model(state_dict_train, model, ability_history_A, ability_history_B, training_step, is_valid=False):
    """
    Fixed version: Compute forward pass without mutating input state,
    only update state_dict_train at the very end after loss computation.
    """
    
    # ========== t=0: Initial state snapshot (NO MUTATION) ==========
    # Create a snapshot of initial state for t=0 computations
    state_t0 = {
        "moneydisposable": state_dict_train["moneydisposable"],
        "savings": state_dict_train["savings"],
        "ability": state_dict_train["ability"],
        "is_superstar_vA": state_dict_train["is_superstar_vA"],
        "is_superstar_vB": state_dict_train["is_superstar_vB"],
        "tax_params": state_dict_train["tax_params"],
        "ret": state_dict_train["ret"]
    }
    
    # Run model at t=0 using snapshot
    model_out_0 = run_network_given_state(state_t0, model=model)
    
    # Transform to get t=0 outcomes (consumption, savings for t=1)
    part_transition_output_t0 = part_transition_transform(
        state=state_t0, 
        model_out=model_out_0, 
        is_init=True, 
        training_step=training_step
    )
    
    # ========== Prepare t=1 state (still NO MUTATION of state_dict_train) ==========
    # Transition abilities for both branches
    ability_next_A, is_superstar_next_A = transition_ability_batched(
        ability=state_t0["ability"],
        is_superstar_prev=state_t0["is_superstar_vA"],
        ability_history=ability_history_A,
        rho_ability=rho_v, sigma_ability=sigma_v,
        p=p if not is_valid else 1, q=q if not is_valid else 1, ability_bar=v_bar,
        v_min=v_min, v_max=v_max,
        zero=is_valid
    )

    ability_next_B, is_superstar_next_B = transition_ability_batched(
        ability=state_t0["ability"],
        is_superstar_prev=state_t0["is_superstar_vB"],
        ability_history=ability_history_B,
        rho_ability=rho_v, sigma_ability=sigma_v,
        p=p if not is_valid else 1, q=q if not is_valid else 1, ability_bar=v_bar,
        v_min=v_min, v_max=v_max,
        zero=is_valid
    )

    # Update ability histories (these are separate tensors, OK to update)
    ability_history_A = update_ability_history(
        ability_history=ability_history_A,
        ability_next=ability_next_A,
        max_len=MAX_HISTORY_LEN
    )
    ability_history_B = update_ability_history(
        ability_history=ability_history_B,
        ability_next=ability_next_B,
        max_len=MAX_HISTORY_LEN
    )
    
    # Build t=1 state for branch A
    state_t1_A = {
        "moneydisposable": part_transition_output_t0["moneydisposable"],
        "savings": part_transition_output_t0["savings"],
        "consumption": part_transition_output_t0["consumption"],
        "wage": part_transition_output_t0["wage"],
        "ret": part_transition_output_t0["ret"],
        "ibt": part_transition_output_t0["ibt"],
        "tax_params": part_transition_output_t0["tax_params"],
        "ability_A": ability_next_A,
        "ability_B": ability_next_B  # Keep both for state construction
    }
    
    # Build t=1 state for branch B (shares same base, different ability)
    state_t1_B = {
        "moneydisposable": part_transition_output_t0["moneydisposable"],
        "savings": part_transition_output_t0["savings"],
        "consumption": part_transition_output_t0["consumption"],
        "wage": part_transition_output_t0["wage"],
        "ret": part_transition_output_t0["ret"],
        "ibt": part_transition_output_t0["ibt"],
        "tax_params": part_transition_output_t0["tax_params"],
        "ability_A": ability_next_A,
        "ability_B": ability_next_B
    }

    # ========== t=1: Forward pass for both branches ==========
    model_out_1A = run_network_given_state(state_t1_A, model=model, brach="A")
    model_out_1B = run_network_given_state(state_t1_B, model=model, brach="B")
    
    part_transition_output_A = part_transition_transform(
        state_t1_A, 
        model_out_1A, 
        branch="A", 
        is_init=False, 
        training_step=training_step
    )
    part_transition_output_B = part_transition_transform(
        state_t1_B, 
        model_out_1B, 
        branch="B", 
        is_init=False, 
        training_step=training_step
    )

    # ========== Loss Computation (clean graph) ==========
    fb_loss = fbloss(
        savings_ratio=model_out_0["savings_t1"], 
        mutilpier=model_out_0["mu_t0"]
    )
    
    aux_loss = auxiloss(
        c0=part_transition_output_t0["consumption"],  # Use t0 output, not state_dict_tmp
        c1_A=part_transition_output_A["consumption"],
        c1_B=part_transition_output_B["consumption"],
        mu_t0=model_out_0["mu_t0"],
        savings2_A=part_transition_output_A["savings"], 
        savings2_B=part_transition_output_B["savings"],  # Fixed: was using A twice
        ibt_A=part_transition_output_A["ibt"], 
        ibt_B=part_transition_output_B["ibt"],
        ret0=state_t0["ret"],  # Use snapshot
        theta=theta
    )
    # aux_loss = auxiloss_stable_v2(c0=part_transition_output_t0["consumption"],
    #          c1_A=part_transition_output_A["consumption"],c1_B=part_transition_output_B["consumption"],
    #          mu_t0=model_out_0["mu_t0"],
    #          savings2_A=part_transition_output_A["savings"], savings2_B=part_transition_output_B["savings"],
    #          ibt_A=part_transition_output_A["ibt"], ibt_B=part_transition_output_B["ibt"],
    #          ret0=state_t0["ret"], theta=theta, mode="log")
    
    laborfoc_loss = laborfocloss_stable(
        labor=model_out_0["labor_t0"],
        consumption=part_transition_output_t0["consumption"],  # Use t0 output
        ibt=part_transition_output_t0["ibt"],  # Use t0 output
        wage=part_transition_output_t0["wage"],  # Use t0 output
        ability=state_t0["ability"],  # Use snapshot
        taxparams=TAX_PARAMS
    )
    
    total_loss = fb_loss + LAMBDA_AUX * aux_loss + LAMBDA_FOC * laborfoc_loss

    # ========== NOW update state_dict_train (AFTER loss computation) ==========
    # This happens outside the computational graph used for loss.backward()
    state_dict_train["is_superstar_vA"] = is_superstar_next_A.detach()
    state_dict_train["is_superstar_vB"] = is_superstar_next_B.detach()
    state_dict_train["moneydisposable"] = part_transition_output_A["moneydisposable"].detach()
    state_dict_train["savings"] = part_transition_output_A["savings"].detach()
    state_dict_train["ability"] = ability_next_A.detach()
    state_dict_train["ret"] = part_transition_output_A["ret"].detach()
    state_dict_train["consumption"] = part_transition_output_A["consumption"].detach()

    return state_dict_train, ability_history_A, ability_history_B, total_loss, {
        'fb_loss': fb_loss.item(),
        'aux_loss': aux_loss.item(),
        'laborfoc_loss': laborfoc_loss.item()
    }

In [44]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5000, gamma=0.5)

In [45]:
wandb.init(
    project="sml-1",
    name=f"{EXP_NAME}_{RUN_NAME}_baseline_mleak",
)

# --- Initialize state ---
state_dict_train = initial_state(required_batch_size=256)
ability_history_A = None
ability_history_B = None
loss_history = []

TRAINING_STEPS = 1000
SAVE_INTERVAL = 300

with tqdm(total=TRAINING_STEPS, desc="Training Progress", ncols=100) as pbar:
    for i in range(TRAINING_STEPS):
        if i % TRAIN_STEP_INTERVAL == 0:

            optimizer.zero_grad()

            # --- Forward ---
            state_dict_train, ability_history_A, ability_history_B, loss, loss_components = run_model(
                state_dict_train=state_dict_train,
                model=model,
                ability_history_A=ability_history_A,
                ability_history_B=ability_history_B,
                training_step=i
            )

            # --- Backward ---
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            # --- Detach ---
            state_dict_train = detach_state_dict(state_dict_train)
            if ability_history_A is not None: ability_history_A = ability_history_A.detach()
            if ability_history_B is not None: ability_history_B = ability_history_B.detach()

            loss_history.append(loss.item())

            # --- Compute mean for each state variable (skip tax_params) ---
            state_means = {}
            for key in ["moneydisposable", "savings", "consumption"]:
                tensor = state_dict_train[key]
                if torch.is_tensor(tensor):
                    state_means[f"state/{key}_mean"] = tensor.mean().item()
                else:
                    state_means[f"state/{key}_mean"] = float(tensor)

            # --- W&B log ---
            wandb.log({
                "step": i,
                "loss": loss.item(),
                "fb_loss": loss_components["fb_loss"],
                "aux_loss": loss_components["aux_loss"],
                "laborfoc_loss": loss_components["laborfoc_loss"],
                "learning_rate": scheduler.get_last_lr()[0],
                **state_means,  # <== merged here
            })

            # --- Periodic save ---
            if i % SAVE_INTERVAL == 0 and i > 0:
                # Save model weights
                weight_path = os.path.join(WEIGHT_DIR, f"model_step_{i}.pt")
                torch.save({
                    "step": i,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "loss": loss.item(),
                }, weight_path)
                print(f"\n✅ Saved model weights: {weight_path}")

                # Save state dict
                state_path = os.path.join(STATE_DIR, f"state_step_{i}.pkl")
                with open(state_path, "wb") as f:
                    pickle.dump(state_dict_train, f)
                print(f"📦 Saved state dict: {state_path}")

            # --- save 4 the first batch ---
            if i == 0:
                # Save state dict
                state_path = os.path.join(STATE_DIR, f"state_step_{i}.pkl")
                with open(state_path, "wb") as f:
                    pickle.dump(state_dict_train, f)
                print(f"📦 Saved state dict: {state_path}")
            # --- Progress bar ---
            pbar.set_postfix({
                "Loss": f"{loss.item():.4e}",
                "FB": f"{loss_components['fb_loss']:.4e}",
                "Aux": f"{loss_components['aux_loss']:.4e}",
                "LaborFOC": f"{loss_components['laborfoc_loss']:.4e}",
                "LR": f"{scheduler.get_last_lr()[0]:.2e}"
            })
        pbar.update(1)

# --- Final save ---
torch.save(model.state_dict(), final_weight_path)
print(f"\n🎯 Final model weights saved to: {final_weight_path}")

with open(final_state_path, "wb") as f:
    pickle.dump(state_dict_train, f)
print(f"📦 Final training state saved to: {final_state_path}")

wandb.finish()


wandb: Currently logged in as: zhinghe78 (zhinghe78-uccu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Training Progress:   0%| | 1/1000 [00:00<04:03,  4.10it/s, Loss=3.7899e-01, FB=4.3394e-02, Aux=3.925

📦 Saved state dict: ./checkpoints/exp_transition/run/states/state_step_0.pkl


Training Progress:  30%|▎| 303/1000 [00:28<01:03, 10.93it/s, Loss=1.7755e-02, FB=1.2138e-03, Aux=1.2


✅ Saved model weights: ./checkpoints/exp_transition/run/weights/model_step_300.pt
📦 Saved state dict: ./checkpoints/exp_transition/run/states/state_step_300.pkl


...

Training Progress:  90%|▉| 903/1000 [01:23<00:08, 11.10it/s, Loss=2.2438e-02, FB=8.7061e-04, Aux=1.0


✅ Saved model weights: ./checkpoints/exp_transition/run/weights/model_step_900.pt
📦 Saved state dict: ./checkpoints/exp_transition/run/states/state_step_900.pkl


Training Progress: 100%|█| 1000/1000 [01:31<00:00, 10.87it/s, Loss=2.7106e-02, FB=1.0194e-03, Aux=1.


🎯 Final model weights saved to: ./checkpoints/exp_transition/run/weights/model_final.pt
📦 Final training state saved to: ./checkpoints/exp_transition/run/states/final_state.pkl


aux_loss,▇█▇▆▆▄▄▄▄▃▃▃▃▃▂▂▂▂▃▂▂▂▂▂▂▂▁▁▂▂▂▁▂▂▂▂▃▄▃▄
fb_loss,█▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
laborfoc_loss,█▃▃▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▂▂▂▂▂▂▂▂▂▂▃▃▅
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss,█▅▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
state/consumption_mean,██▇▆▇▇▇█▆▆▅▄▄▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▂▂▁▂▂▁▁▁▁▂▂▃
state/moneydisposable_mean,▁▄▆▇▇▇▇▇▇▇▇█████████████████████████████
state/savings_mean,▁▂▃▅▇▇▆▆▆▆▇▇▇▇▇█▇▇█████████████████████▇
step,▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇█████
aux_loss,0.00124
fb_loss,0.00102


# Plot decision rule

In [46]:
# Load model
checkpoint = torch.load(final_weight_path, map_location="cuda" if torch.cuda.is_available() else "mps")
if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)
model.eval()

# Load state
with open("/Users/chenjinghe/Desktop/python-projects/SML-2/checkpoints/exp_transition/run/states/state_step_0.pkl", "rb") as f:
    state_dict_train = pickle.load(f)

with open(final_state_path, "rb") as f:
    state_dict_tail = pickle.load(f)

In [75]:
DECISION_RULE_PLOT_POINTS = 100
DECISION_RULE_Y_REALIZATIONS = 7 # CHANGE NAME TO Y

def dim(x, i): # Add dimension to x at axis i helper function
    return np.expand_dims(x, axis=i)


def plot_init_batch(state_dict_train, batch_number):
    state_dict_train_batch = {}
    for keys in state_dict_train.keys():
        state_dict_train_batch[keys] = state_dict_train[keys][batch_number]

    return state_dict_train_batch

# Use the first batch
batch_number = 0
state_dict_train_batch = plot_init_batch(state_dict_train=state_dict_train, batch_number=0)

In [79]:
moneydisposable_valid = state_dict_train_batch["moneydisposable"]
moneydisposable_valid  = np.repeat(dim(dim(moneydisposable_valid , 0), 0), DECISION_RULE_PLOT_POINTS, axis=0) 
moneydisposable_valid[:, :, 0] = dim(np.linspace(
    state_dict_tail["moneydisposable"].min(), state_dict_tail["moneydisposable"].max(), 
    DECISION_RULE_PLOT_POINTS), 1)
moneydisposable_valid  = np.repeat(moneydisposable_valid , DECISION_RULE_Y_REALIZATIONS, axis=1)



ability_valid  = state_dict_train_batch["ability"]
ability_valid  = np.repeat(dim(ability_valid, 0), DECISION_RULE_Y_REALIZATIONS, axis=0)
ability_valid[:, 0] = np.linspace(state_dict_tail["ability"].min(), 
                            state_dict_tail["ability"].max(), DECISION_RULE_Y_REALIZATIONS)

tax_params_valid = np.repeat(dim(state_dict_train_batch["tax_params"], 0), DECISION_RULE_Y_REALIZATIONS, axis=0)

In [84]:
decision_rule_results = {
    "m0": np.empty(shape=(DECISION_RULE_PLOT_POINTS, DECISION_RULE_Y_REALIZATIONS, AGENTS)),
    "m1": np.empty(shape=(DECISION_RULE_PLOT_POINTS, DECISION_RULE_Y_REALIZATIONS, AGENTS)),
    "c0": np.empty(shape=(DECISION_RULE_PLOT_POINTS, DECISION_RULE_Y_REALIZATIONS, AGENTS)),
    "a1": np.empty(shape=(DECISION_RULE_PLOT_POINTS, DECISION_RULE_Y_REALIZATIONS, AGENTS)),
    "h0": np.empty(shape=(DECISION_RULE_PLOT_POINTS, DECISION_RULE_Y_REALIZATIONS, AGENTS)),
}

state_dict_valid = build_inputs(
    moneydisposable=moneydisposable_valid[0],
    ability=ability_valid,
    tax_params=tax_params_valid    
)

In [ ]:
ability_history_A_valid = None
ability_history_B_valid = None

for step in DECISION_RULE_PLOT_POINTS:
    state_dict_valid, _, _, _, _ = run_model(
        state_dict_train=state_dict_valid, 
        model = model,
        ability_history_A=ability_history_A_valid,
        ability_history_B=ability_history_B_valid,
        
    )